# Assignment 2: Transformer Language Models

Replace the LSTM from A1 with a Transformer (matching OLMo-2 architecture), then generate text.

Run each cell in sequence.

## Step 0: Preliminaries

In [1]:
# Step 0: Imports and reuse A1 components
import torch, nltk, pickle, math, time, os, sys
from torch import nn
import torch.nn.functional as F
from collections import Counter
from transformers import BatchEncoding, PretrainedConfig, PreTrainedModel, TrainingArguments
from transformers.modeling_outputs import CausalLMOutput
from torch.utils.data import DataLoader
from datasets import load_dataset
import numpy as np

nltk.download('punkt_tab', quiet=True)

# Paths — A1 data is one directory up
TRAIN_FILE = '../assignment1/train.txt'
VAL_FILE = '../assignment1/val.txt'
OUTPUT_DIR = 'trained_model'

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
print(f'MPS: {torch.backends.mps.is_available()}')

PyTorch: 2.11.0
CUDA: False
MPS: True


In [2]:
# Reuse A1 tokenizer and trainer
# (copy the classes here so this notebook is self-contained)

def lowercase_tokenizer(text):
    return [t.lower() for t in nltk.word_tokenize(text)]


class A1Tokenizer:
    def __init__(self, str_to_int, tokenize_fun, pad_token, unk_token, bos_token, eos_token, model_max_length=None):
        self.str_to_int = str_to_int
        self.int_to_str = {i: s for s, i in str_to_int.items()}
        self.tokenize_fun = tokenize_fun
        self.pad_token = pad_token
        self.unk_token = unk_token
        self.bos_token = bos_token
        self.eos_token = eos_token
        self.pad_token_id = str_to_int[pad_token]
        self.unk_token_id = str_to_int[unk_token]
        self.bos_token_id = str_to_int[bos_token]
        self.eos_token_id = str_to_int[eos_token]
        self.model_max_length = model_max_length

    def __call__(self, texts, truncation=False, padding=False, return_tensors=None):
        if return_tensors and return_tensors != 'pt':
            raise ValueError('Should be pt')
        all_ids = []
        for text in texts:
            tokens = self.tokenize_fun(text)
            ids = ([self.bos_token_id]
                   + [self.str_to_int.get(t, self.unk_token_id) for t in tokens]
                   + [self.eos_token_id])
            if truncation and self.model_max_length is not None:
                ids = ids[:self.model_max_length]
            all_ids.append(ids)
        if padding:
            max_len = max(len(ids) for ids in all_ids)
            attention_mask = []
            for i, ids in enumerate(all_ids):
                pad_len = max_len - len(ids)
                attention_mask.append([1] * len(ids) + [0] * pad_len)
                all_ids[i] = ids + [self.pad_token_id] * pad_len
        else:
            attention_mask = [[1] * len(ids) for ids in all_ids]
        if return_tensors == 'pt':
            all_ids = torch.tensor(all_ids, dtype=torch.long)
            attention_mask = torch.tensor(attention_mask, dtype=torch.long)
        return BatchEncoding({'input_ids': all_ids, 'attention_mask': attention_mask})

    def __len__(self):
        return len(self.str_to_int)

    def save(self, filename):
        with open(filename, 'wb') as f:
            pickle.dump(self, f)

    @staticmethod
    def from_file(filename):
        with open(filename, 'rb') as f:
            return pickle.load(f)


def build_tokenizer(train_file, tokenize_fun=lowercase_tokenizer, max_voc_size=None, model_max_length=None,
                    pad_token='<PAD>', unk_token='<UNK>', bos_token='<BOS>', eos_token='<EOS>'):
    counter = Counter()
    with open(train_file) as f:
        for line in f:
            line = line.strip()
            if line:
                tokens = tokenize_fun(line)
                counter.update(tokens)
    special_tokens = [pad_token, unk_token, bos_token, eos_token]
    if max_voc_size is not None:
        n_regular = max_voc_size - len(special_tokens)
        most_common = [tok for tok, _ in counter.most_common(n_regular)]
    else:
        most_common = [tok for tok, _ in counter.most_common()]
    vocab = special_tokens + most_common
    str_to_int = {tok: i for i, tok in enumerate(vocab)}
    return A1Tokenizer(
        str_to_int=str_to_int, tokenize_fun=tokenize_fun,
        pad_token=pad_token, unk_token=unk_token,
        bos_token=bos_token, eos_token=eos_token,
        model_max_length=model_max_length,
    )

print('A1 components loaded.')

A1 components loaded.


In [3]:
# Build tokenizer and load data (reuse from A1)
tokenizer = build_tokenizer(TRAIN_FILE, max_voc_size=10000, model_max_length=128)
print(f'Vocabulary size: {len(tokenizer)}')

dataset = load_dataset('text', data_files={'train': TRAIN_FILE, 'val': VAL_FILE})
dataset = dataset.filter(lambda x: x['text'].strip() != '')
print(f'Train: {len(dataset["train"]):,}  Val: {len(dataset["val"]):,}')

Vocabulary size: 10000


Generating train split: 0 examples [00:00, ? examples/s]

Generating val split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/294118 [00:00<?, ? examples/s]

Filter:   0%|          | 0/35748 [00:00<?, ? examples/s]

Train: 147,059  Val: 17,874


---
## Step 1: Setting Up a Transformer Neural Network

### Configuration

In [4]:
# Model configuration (matches OLMo-2 structure)

class A2ModelConfig(PretrainedConfig):
    """Configuration for the Transformer language model."""
    def __init__(self, vocab_size=None, hidden_size=None, intermediate_size=None,
                 num_attention_heads=None, num_hidden_layers=None,
                 rope_theta=None, hidden_act='silu',
                 max_position_embeddings=None, rms_norm_eps=None, **kwargs):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.max_position_embeddings = max_position_embeddings
        self.rms_norm_eps = rms_norm_eps
        self.num_attention_heads = num_attention_heads
        self.rope_theta = rope_theta
        self.hidden_act = hidden_act
        self.intermediate_size = intermediate_size
        self.num_hidden_layers = num_hidden_layers

print('A2ModelConfig defined.')

A2ModelConfig defined.


### Task 1.1: MLP Layer (SwiGLU)

In [5]:
# Task 1.1: SwiGLU MLP
#
# SwiGLU(x) = (SiLU(xW_gate) * xW_up) W_down
#
# Three linear layers (all bias=False):
#   gate_proj: hidden_size -> intermediate_size
#   up_proj:   hidden_size -> intermediate_size
#   down_proj: intermediate_size -> hidden_size

class A2MLP(nn.Module):
    """SwiGLU MLP layer."""
    def __init__(self, config):
        super().__init__()
        assert config.hidden_act == 'silu'
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.up_proj   = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)
        self.act_fn = nn.SiLU()

    def forward(self, hidden_states):
        return self.down_proj(self.act_fn(self.gate_proj(hidden_states)) * self.up_proj(hidden_states))


# Sanity check
_cfg = A2ModelConfig(vocab_size=100, hidden_size=64, intermediate_size=128,
                     num_attention_heads=4, num_hidden_layers=2,
                     rope_theta=10000.0, rms_norm_eps=1e-5)
_mlp = A2MLP(_cfg)
_x = torch.randn(2, 10, 64)
_out = _mlp(_x)
print(f'MLP: input {_x.shape} -> output {_out.shape}')
assert _out.shape == _x.shape, 'Shape mismatch!'
print('Task 1.1 PASS')

MLP: input torch.Size([2, 10, 64]) -> output torch.Size([2, 10, 64])
Task 1.1 PASS


### Task 1.2: RMSNorm

In [6]:
# Task 1.2: RMS Layer Normalization
# Using PyTorch's built-in RMSNorm

class A2RMSNorm(nn.Module):
    """RMS layer normalization."""
    def __init__(self, config):
        super().__init__()
        self.norm = nn.RMSNorm(config.hidden_size, eps=config.rms_norm_eps, elementwise_affine=True)

    def forward(self, hidden_states):
        return self.norm(hidden_states)


# Sanity check
_norm = A2RMSNorm(_cfg)
_out = _norm(_x)
print(f'RMSNorm: input {_x.shape} -> output {_out.shape}')
assert _out.shape == _x.shape, 'Shape mismatch!'
print('Task 1.2 PASS')

RMSNorm: input torch.Size([2, 10, 64]) -> output torch.Size([2, 10, 64])
Task 1.2 PASS


### Task 1.3: Multi-Head Attention

In [7]:
# Task 1.3: Multi-Head Attention with RoPE
#
# Steps:
#   1. Project: Q = xW_Q, K = xW_K, V = xW_V
#   2. Normalize Q, K with RMSNorm
#   3. Reshape: (b, m, d) -> (b, n_h, m, d_h)
#   4. Apply RoPE rotations to Q, K
#   5. Scaled dot-product attention with causal mask
#   6. Reshape back: (b, n_h, m, d_h) -> (b, m, d)
#   7. Output projection: W_O

class A2Attention(nn.Module):
    """Multi-head attention with RoPE and causal masking."""

    def __init__(self, config):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.head_dim = config.hidden_size // config.num_attention_heads
        assert self.head_dim * self.num_heads == self.hidden_size

        # Q, K, V, O projections (all bias=False)
        self.q_proj = nn.Linear(self.hidden_size, self.hidden_size, bias=False)
        self.k_proj = nn.Linear(self.hidden_size, self.hidden_size, bias=False)
        self.v_proj = nn.Linear(self.hidden_size, self.hidden_size, bias=False)
        self.o_proj = nn.Linear(self.hidden_size, self.hidden_size, bias=False)

        # RMSNorm after Q and K projections
        self.q_norm = A2RMSNorm(config)
        self.k_norm = A2RMSNorm(config)

    def forward(self, hidden_states, rope_rotations):
        b, m, d = hidden_states.shape
        n_h, d_h = self.num_heads, self.head_dim

        # Step 1: Project
        q = self.q_proj(hidden_states)
        k = self.k_proj(hidden_states)
        v = self.v_proj(hidden_states)

        # Step 2: Normalize Q and K
        q = self.q_norm(q)
        k = self.k_norm(k)

        # Step 3: Reshape to (b, n_h, m, d_h)
        q = q.view(b, m, n_h, d_h).transpose(1, 2)
        k = k.view(b, m, n_h, d_h).transpose(1, 2)
        v = v.view(b, m, n_h, d_h).transpose(1, 2)

        # Step 4: Apply RoPE
        q, k = apply_rotary_pos_emb(q, k, rope_rotations)

        # Step 5: Scaled dot-product attention with causal mask
        attn_out = F.scaled_dot_product_attention(q, k, v, is_causal=True)

        # Step 6: Reshape back to (b, m, d)
        attn_out = attn_out.transpose(1, 2).reshape(b, m, d)

        # Step 7: Output projection
        return self.o_proj(attn_out)


print('A2Attention defined.')

A2Attention defined.


In [8]:
# RoPE implementation (from skeleton — pre-built)

def apply_rotary_pos_emb(q, k, rope_rotations, unsqueeze_dim=1):
    """Applies precomputed RoPE rotations to query and key."""
    assert q.shape == k.shape
    assert len(q.shape) == 4
    cos, sin = rope_rotations
    assert q.shape[2] == cos.shape[1]
    assert q.shape[3] == cos.shape[2]
    q_type, k_type = q.dtype, k.dtype
    cos = cos.unsqueeze(unsqueeze_dim)
    sin = sin.unsqueeze(unsqueeze_dim)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed.to(q_type), k_embed.to(k_type)

def rotate_half(x):
    """Rotates half the hidden dims of the input."""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

class A2RotaryEmbedding(nn.Module):
    """RoPE position representation."""
    def __init__(self, config, device=None):
        super().__init__()
        rope_theta = config.rope_theta
        head_dim = config.hidden_size // config.num_attention_heads
        partial_rotary_factor = 1.0
        dim = int(head_dim * partial_rotary_factor)
        self.inv_freq = 1.0 / (rope_theta ** (torch.arange(0, dim, 2, dtype=torch.int64).to(device=device, dtype=torch.float) / dim))

    @torch.no_grad()
    def forward(self, x):
        position_ids = torch.arange(0, x.shape[1], device=x.device).unsqueeze(0)
        inv_freq_expanded = self.inv_freq[None, :, None].float().expand(position_ids.shape[0], -1, 1).to(x.device)
        position_ids_expanded = position_ids[:, None, :].float()
        device_type = x.device.type if isinstance(x.device.type, str) and x.device.type != 'mps' else 'cpu'
        with torch.autocast(device_type=device_type, enabled=False):
            freqs = (inv_freq_expanded.float() @ position_ids_expanded.float()).transpose(1, 2)
            emb = torch.cat((freqs, freqs), dim=-1)
            cos = emb.cos()
            sin = emb.sin()
            return cos, sin

print('RoPE components defined.')

RoPE components defined.


In [9]:
# Sanity check: Attention
_rope = A2RotaryEmbedding(_cfg)
_attn = A2Attention(_cfg)
_test_ids = torch.randint(0, 100, (2, 10))
_rope_rot = _rope(_test_ids)
_out = _attn(_x, _rope_rot)
print(f'Attention: input {_x.shape} -> output {_out.shape}')
assert _out.shape == _x.shape, 'Shape mismatch!'
print('Task 1.3 PASS')

Attention: input torch.Size([2, 10, 64]) -> output torch.Size([2, 10, 64])
Task 1.3 PASS


### Task 1.4: Full Transformer Decoder Layer

In [11]:
# Task 1.4: Decoder layer = Attention + MLP + Norms + Residuals
#
# Pre-norm architecture (Llama-style):
#   h = attention(norm(x)) + x       [residual around attention]
#   out = mlp(norm(h)) + h           [residual around MLP]

class A2DecoderLayer(nn.Module):
    """A complete Transformer decoder layer."""
    def __init__(self, config):
        super().__init__()
        self.self_attn = A2Attention(config)
        self.mlp = A2MLP(config)
        self.input_layernorm = A2RMSNorm(config)     # pre-attention norm
        self.post_attention_layernorm = A2RMSNorm(config)  # pre-MLP norm

    def forward(self, hidden_states, rope_rotations):
        # Attention block with residual
        residual = hidden_states
        hidden_states = self.input_layernorm(hidden_states)
        hidden_states = self.self_attn(hidden_states, rope_rotations)
        hidden_states = hidden_states + residual

        # MLP block with residual
        residual = hidden_states
        hidden_states = self.post_attention_layernorm(hidden_states)
        hidden_states = self.mlp(hidden_states)
        hidden_states = hidden_states + residual

        return hidden_states


# Sanity check
_layer = A2DecoderLayer(_cfg)
_out = _layer(_x, _rope_rot)
print(f'DecoderLayer: input {_x.shape} -> output {_out.shape}')
assert _out.shape == _x.shape, 'Shape mismatch!'
print('Task 1.4 PASS')

DecoderLayer: input torch.Size([2, 10, 64]) -> output torch.Size([2, 10, 64])
Task 1.4 PASS


### Task 1.5: Complete Transformer Stack

In [12]:
# Task 1.5: Full Transformer language model
#
# Architecture:
#   Embedding -> [DecoderLayer x N] -> Final RMSNorm -> Unembedding
#
# RoPE rotations generated once, passed to all layers.

class A2Transformer(PreTrainedModel):
    """Transformer language model (OLMo-2 architecture)."""
    config_class = A2ModelConfig

    def __init__(self, config):
        super().__init__(config)

        self.rotary_emb = A2RotaryEmbedding(config)
        self.embedding = nn.Embedding(config.vocab_size, config.hidden_size)
        self.layers = nn.ModuleList(
            [A2DecoderLayer(config) for _ in range(config.num_hidden_layers)]
        )
        self.final_norm = A2RMSNorm(config)
        self.unembedding = nn.Linear(config.hidden_size, config.vocab_size, bias=False)

        self.loss_func = torch.nn.CrossEntropyLoss(ignore_index=-100)

        self.post_init()

    def forward(self, input_ids, labels=None):
        rope_rotations = self.rotary_emb(input_ids)

        hidden_states = self.embedding(input_ids)
        for layer in self.layers:
            hidden_states = layer(hidden_states, rope_rotations)
        hidden_states = self.final_norm(hidden_states)
        logits = self.unembedding(hidden_states)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = self.loss_func(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
            )

        return CausalLMOutput(logits=logits, loss=loss)


print('A2Transformer defined.')

A2Transformer defined.


In [13]:
# Sanity check: full model
_full_cfg = A2ModelConfig(
    vocab_size=100, hidden_size=64, intermediate_size=128,
    num_attention_heads=4, num_hidden_layers=2,
    rope_theta=10000.0, rms_norm_eps=1e-5,
    max_position_embeddings=128,
)
_model = A2Transformer(_full_cfg)
_ids = torch.randint(0, 100, (2, 10))
_out = _model(input_ids=_ids)
print(f'Full model: input {_ids.shape} -> logits {_out.logits.shape}')
assert _out.logits.shape == (2, 10, 100), 'Shape mismatch!'

_out_loss = _model(input_ids=_ids, labels=_ids)
expected = math.log(100)
print(f'Initial loss: {_out_loss.loss.item():.4f} (expected ~{expected:.4f})')
print('Task 1.5 PASS')

Full model: input torch.Size([2, 10]) -> logits torch.Size([2, 10, 100])
Initial loss: 4.6194 (expected ~4.6052)
Task 1.5 PASS


---
## Step 2: Training the Language Model

### Task 2.1: Training

In [14]:
# Trainer (reused from A1 with minor adaptation)

class A2Trainer:
    def __init__(self, model, args, train_dataset, eval_dataset, tokenizer):
        self.model = model
        self.args = args
        self.train_dataset = train_dataset
        self.eval_dataset = eval_dataset
        self.tokenizer = tokenizer

    def select_device(self):
        if self.args.use_cpu:
            return torch.device('cpu')
        if torch.cuda.is_available():
            return torch.device('cuda')
        if torch.mps.is_available():
            return torch.device('mps')
        return torch.device('cpu')

    def _collate_fn(self, batch):
        texts = [item['text'] for item in batch]
        encoded = self.tokenizer(texts, truncation=True, padding=True, return_tensors='pt')
        return encoded['input_ids']

    def _evaluate(self, val_loader, device):
        self.model.eval()
        total_loss, n = 0.0, 0
        with torch.no_grad():
            for input_ids in val_loader:
                input_ids = input_ids.to(device)
                labels = input_ids.clone()
                labels[labels == self.tokenizer.pad_token_id] = -100
                output = self.model(input_ids=input_ids, labels=labels)
                total_loss += output.loss.item()
                n += 1
        avg_loss = total_loss / n
        return avg_loss, math.exp(avg_loss)

    def train(self):
        args = self.args
        device = self.select_device()
        print(f'Device: {device}')
        self.model.to(device)

        optimizer = torch.optim.AdamW(self.model.parameters(), lr=args.learning_rate)
        train_loader = DataLoader(self.train_dataset, batch_size=args.per_device_train_batch_size,
                                  shuffle=True, collate_fn=self._collate_fn)
        val_loader = DataLoader(self.eval_dataset, batch_size=args.per_device_eval_batch_size,
                                shuffle=False, collate_fn=self._collate_fn)

        for epoch in range(int(args.num_train_epochs)):
            self.model.train()
            running_loss, n = 0.0, 0
            t0 = time.time()
            for input_ids in train_loader:
                input_ids = input_ids.to(device)
                labels = input_ids.clone()
                labels[labels == self.tokenizer.pad_token_id] = -100
                output = self.model(input_ids=input_ids, labels=labels)
                loss = output.loss
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                running_loss += loss.item()
                n += 1
            val_loss, val_ppl = self._evaluate(val_loader, device)
            print(f'Epoch {epoch+1}/{int(args.num_train_epochs)} | '
                  f'train_loss={running_loss/n:.4f} | val_loss={val_loss:.4f} | '
                  f'val_ppl={val_ppl:.1f} | time={time.time()-t0:.1f}s')

        os.makedirs(args.output_dir, exist_ok=True)
        self.model.save_pretrained(args.output_dir)
        print(f'Saved to {args.output_dir}.')

print('A2Trainer defined.')

A2Trainer defined.


In [15]:
# Create model and train
# Small Transformer: 2 layers, hidden=128, 4 heads, intermediate=256

config = A2ModelConfig(
    vocab_size=len(tokenizer),
    hidden_size=128,
    intermediate_size=256,
    num_attention_heads=4,
    num_hidden_layers=2,
    rope_theta=10000.0,
    rms_norm_eps=1e-5,
    max_position_embeddings=128,
)

model = A2Transformer(config)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {config.num_hidden_layers} layers, hidden={config.hidden_size}, '
      f'heads={config.num_attention_heads}, params={n_params:,}')

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    optim='adamw_torch',
    eval_strategy='epoch',
    learning_rate=5e-4,
    num_train_epochs=3,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    report_to='none',
)

trainer = A2Trainer(
    model=model, args=training_args,
    train_dataset=dataset['train'], eval_dataset=dataset['val'],
    tokenizer=tokenizer,
)
trainer.train()

Model: 2 layers, hidden=128, heads=4, params=2,888,832
Device: mps
Epoch 1/3 | train_loss=4.8662 | val_loss=4.3598 | val_ppl=78.2 | time=736.8s
Epoch 2/3 | train_loss=4.2395 | val_loss=4.1715 | val_ppl=64.8 | time=857.3s
Epoch 3/3 | train_loss=4.0709 | val_loss=4.1047 | val_ppl=60.6 | time=813.8s


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to trained_model.


---
## Step 3: Generating Text

### Task 3.1: Predicting the next word

In [16]:
# Task 3.1: Next-word prediction (greedy)

def predict_next_word(model, tokenizer, text, k=10, device=None):
    if device is None:
        device = next(model.parameters()).device
    model.eval()
    encoded = tokenizer([text], truncation=True, return_tensors='pt')
    input_ids = encoded['input_ids'].to(device)
    with torch.no_grad():
        output = model(input_ids=input_ids)
    last_logits = output.logits[0, -1, :]
    probs = torch.softmax(last_logits, dim=0)
    topk = torch.topk(probs, k)
    return [(tokenizer.int_to_str.get(idx.item(), '<???>'), score.item())
            for score, idx in zip(topk.values, topk.indices)]


device = next(model.parameters()).device
for prompt in ['She lives in San', 'The president of the United',
               'I like to eat', 'The weather is']:
    preds = predict_next_word(model, tokenizer, prompt, k=5, device=device)
    top_words = ', '.join(f'{w} ({s:.3f})' for w, s in preds)
    print(f'"{prompt}"  ->  {top_words}')

"She lives in San"  ->  , (0.142), and (0.138), in (0.136), at (0.043), on (0.039)
"The president of the United"  ->  church (0.194), <UNK> (0.083), conference (0.031), book (0.025), and (0.022)
"I like to eat"  ->  , (0.192), and (0.158), . (0.047), i (0.038), '' (0.031)
"The weather is"  ->  <UNK> (0.072), `` (0.061), and (0.031), that (0.027), <EOS> (0.025)


### Task 3.2: Text generation with sampling

In [17]:
# Task 3.2: Text generation — random sampling with temperature and top-k
#
# Algorithm:
#   1. Encode prompt
#   2. Loop until EOS or max_length:
#      a. Forward pass -> logits at last position
#      b. Scale by temperature
#      c. Keep only top-k logits (set rest to -inf)
#      d. Sample from Categorical distribution
#      e. Append sampled token
#   3. Decode back to text

def generate(model, tokenizer, prompt, max_length=100, temperature=1.0, topk=50, device=None):
    """Generate text using random sampling with temperature and top-k."""
    if device is None:
        device = next(model.parameters()).device

    model.eval()
    encoded = tokenizer([prompt], truncation=True, return_tensors='pt')
    input_ids = encoded['input_ids'].to(device)

    generated = input_ids[0].tolist()  # start with prompt tokens

    with torch.no_grad():
        for _ in range(max_length):
            # Forward pass
            ids_tensor = torch.tensor([generated], dtype=torch.long, device=device)
            # Truncate if exceeding model_max_length
            if tokenizer.model_max_length and len(generated) > tokenizer.model_max_length:
                ids_tensor = ids_tensor[:, -tokenizer.model_max_length:]
            output = model(input_ids=ids_tensor)
            logits = output.logits[0, -1, :]  # last position

            # Temperature scaling
            logits = logits / temperature

            # Top-k filtering
            if topk is not None and topk > 0:
                top_values, top_indices = torch.topk(logits, topk)
                mask = torch.full_like(logits, float('-inf'))
                mask.scatter_(0, top_indices, top_values)
                logits = mask

            # Sample
            dist = torch.distributions.Categorical(logits=logits)
            next_token = dist.sample().item()

            # Stop on EOS
            if next_token == tokenizer.eos_token_id:
                break

            generated.append(next_token)

    # Decode: skip BOS, convert IDs to words
    words = [tokenizer.int_to_str.get(t, '<UNK>') for t in generated
             if t not in (tokenizer.bos_token_id, tokenizer.eos_token_id, tokenizer.pad_token_id)]
    return ' '.join(words)


print('generate() defined.')

generate() defined.


In [18]:
# Generate with different prompts and parameters

test_prompts = [
    'In natural language processing, a Transformer',
    'Is Stockholm the capital of Sweden? Answer yes or no. The answer is',
    'Write a Python program that reverses a list.',
]

print('=== Temperature=0.8, top-k=50 ===')
for prompt in test_prompts:
    text = generate(model, tokenizer, prompt, max_length=50, temperature=0.8, topk=50, device=device)
    print(f'\nPrompt: "{prompt}"')
    print(f'Output: {text}')

print('\n\n=== Temperature=0.3, top-k=10 (more focused) ===')
for prompt in test_prompts:
    text = generate(model, tokenizer, prompt, max_length=50, temperature=0.3, topk=10, device=device)
    print(f'\nPrompt: "{prompt}"')
    print(f'Output: {text}')

print('\n\n=== Temperature=1.5, top-k=100 (more random) ===')
for prompt in test_prompts:
    text = generate(model, tokenizer, prompt, max_length=50, temperature=1.5, topk=100, device=device)
    print(f'\nPrompt: "{prompt}"')
    print(f'Output: {text}')

=== Temperature=0.8, top-k=50 ===

Prompt: "In natural language processing, a Transformer"
Output: in natural language processing , a <UNK> language is a <UNK> tool that is the language of a language , and is that of <UNK> linguistic behavior and <UNK> has been <UNK> with most of the dialects and <UNK> .

Prompt: "Is Stockholm the capital of Sweden? Answer yes or no. The answer is"
Output: is stockholm the capital of sweden ? answer yes or no . the answer is . these <UNK> are `` i <UNK> no <UNK> '' . these <UNK> are <UNK> and are more important to <UNK> , and <UNK> a few years old , at the beginning of the 20th century . the only way about what is now referred to as a <UNK> ,

Prompt: "Write a Python program that reverses a list."
Output: write a <UNK> program that <UNK> a list . , `` the <UNK> '' , `` <UNK> '' , `` <UNK> '' , the student 's <UNK> `` <UNK> '' as an <UNK> ( <UNK> ) .


=== Temperature=0.3, top-k=10 (more focused) ===

Prompt: "In natural language processing, a Transform

### Task 3.3: Comparing to pre-trained OLMo-2 1B

In [19]:
# Task 3.3: Load OLMo-2 1B and compare generation

from transformers import AutoTokenizer, AutoModelForCausalLM

olmo_name = 'allenai/OLMo-2-0425-1B'
print(f'Loading {olmo_name}...')
olmo_tokenizer = AutoTokenizer.from_pretrained(olmo_name)
olmo_model = AutoModelForCausalLM.from_pretrained(olmo_name, torch_dtype=torch.float32)
olmo_model.eval()
print(f'OLMo-2 loaded. Parameters: {sum(p.numel() for p in olmo_model.parameters()):,}')

Loading allenai/OLMo-2-0425-1B...


config.json:   0%|          | 0.00/623 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/179 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

OLMo-2 loaded. Parameters: 1,484,916,736


In [20]:
# Generate with OLMo-2 and compare

def generate_olmo(model, tokenizer, prompt, max_length=50, temperature=0.8, topk=50):
    """Generate text with a HuggingFace CausalLM model."""
    inputs = tokenizer(prompt, return_tensors='pt')
    input_ids = inputs['input_ids']
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_length,
            temperature=temperature,
            top_k=topk,
            do_sample=True,
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


print('=== Comparison: Our model vs OLMo-2 1B ===')
print('(temperature=0.8, top-k=50)\n')

for prompt in test_prompts:
    print(f'Prompt: "{prompt}"')
    our_text = generate(model, tokenizer, prompt, max_length=50, temperature=0.8, topk=50, device=device)
    olmo_text = generate_olmo(olmo_model, olmo_tokenizer, prompt, max_length=50, temperature=0.8, topk=50)
    print(f'  [Ours]   {our_text}')
    print(f'  [OLMo-2] {olmo_text}')
    print()

=== Comparison: Our model vs OLMo-2 1B ===
(temperature=0.8, top-k=50)

Prompt: "In natural language processing, a Transformer"
  [Ours]   in natural language processing , a <UNK> of <UNK> <UNK> was used to <UNK> <UNK> , or <UNK> ( <UNK> ) system .
  [OLMo-2] In natural language processing, a Transformer is a neural network architecture. It was proposed by Vaswani et al. (2017) to perform various natural language processing tasks. The Transformer has replaced the traditional neural network architecture such as the multilayer perceptron for performing various tasks like

Prompt: "Is Stockholm the capital of Sweden? Answer yes or no. The answer is"
  [Ours]   is stockholm the capital of sweden ? answer yes or no . the answer is by saying that the <UNK> would not be true but only it is not true and that it must be possible to <UNK> any decision to be given .
  [OLMo-2] Is Stockholm the capital of Sweden? Answer yes or no. The answer is yes.

Is Stockholm a state of Sweden? Answer yes or n

---
**Done!** All steps complete.